# Phase 3b: PubMed Medical Summarization

AutoDL H800 80GB — 医疗摘要实验（跨领域验证）

| # | 实验 | 方法 | 数据集 | 模型 |
|---|------|------|--------|------|
| E15 | LoRA PubMed Qwen    | LoRA (r=16) | PubMed Summ | Qwen2.5-1.5B |
| E16 | LoRA PubMed Llama   | LoRA (r=16) | PubMed Summ | Llama-3.2-1B |
| E17 | Full FT PubMed Qwen | Full FT     | PubMed Summ | Qwen2.5-1.5B |
| E18 | Full FT PubMed Llama| Full FT     | PubMed Summ | Llama-3.2-1B |
| E19 | Random PubMed Qwen  | LoRA (shuffled labels) | PubMed Summ | Qwen2.5-1.5B |
| E20 | Random PubMed Llama | LoRA (shuffled labels) | PubMed Summ | Llama-3.2-1B |
| B3  | Baseline PubMed Qwen | Zero-shot  | PubMed Summ | Qwen2.5-1.5B |
| B4  | Baseline PubMed Llama| Zero-shot  | PubMed Summ | Llama-3.2-1B |

**模型加载策略：** 所有模型（Qwen、Llama、roberta-large）先通过 `snapshot_download` 下载到本地 HF cache，之后训练和推理全部从本地缓存加载，不依赖网络。

## 0. 环境准备

In [1]:
import os, sys
os.chdir('/root/MLP')
os.environ['HF_HOME'] = '/root/autodl-tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/root/autodl-tmp/hf_cache'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['WANDB_MODE'] = 'offline'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.makedirs('/root/autodl-tmp/hf_cache', exist_ok=True)
os.makedirs('/root/autodl-tmp/outputs', exist_ok=True)
os.makedirs('/root/autodl-tmp/logs', exist_ok=True)
os.makedirs('/root/autodl-tmp/data', exist_ok=True)

# 软链到数据盘
for target, link in [('/root/autodl-tmp/outputs', '/root/MLP/outputs'),
                      ('/root/autodl-tmp/logs',    '/root/MLP/logs'),
                      ('/root/autodl-tmp/data',    '/root/MLP/data')]:
    if not os.path.islink(link):
        if os.path.isdir(link):
            print(f'  ⚠ {link} is a real dir, skipping')
        else:
            os.symlink(target, link)
            print(f'  ✓ created: {link} -> {target}')
    else:
        print(f'  ✓ symlink exists: {link} -> {os.readlink(link)}')

!pwd && ls
print(f'Python: {sys.executable}')
print(f'HF_HOME: {os.environ["HF_HOME"]}')
print(f'HF_ENDPOINT: {os.environ["HF_ENDPOINT"]}')
print(f'WANDB_MODE: {os.environ["WANDB_MODE"]}')

  ✓ symlink exists: /root/MLP/outputs -> /root/autodl-tmp/outputs
  ⚠ /root/MLP/logs is a real dir, skipping
  ✓ symlink exists: /root/MLP/data -> /root/autodl-tmp/data
/root/MLP
README.md			  configs  results
autodl_run.ipynb		  data	   roberta-large
autodl_run_phase2.ipynb		  logs	   scripts
autodl_run_phase3_nfcorpus.ipynb  models   src
autodl_run_phase3_pubmed.ipynb	  outputs  wandb
Python: /root/miniconda3/bin/python
HF_HOME: /root/autodl-tmp/hf_cache
HF_ENDPOINT: https://hf-mirror.com
WANDB_MODE: offline


In [2]:
import sys
!{sys.executable} -m pip install peft accelerate trl bitsandbytes wandb rouge-score bert-score scikit-learn sentencepiece huggingface_hub datasets
print('\n✅ 安装完成！如果是第一次装，请重启内核：Kernel → Restart Kernel')

Looking in indexes: http://mirrors.aliyun.com/pypi/simple

✅ 安装完成！如果是第一次装，请重启内核：Kernel → Restart Kernel


In [3]:
import torch, peft, trl
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA H800 PCIe


In [4]:
# HuggingFace 登录 (Llama 需要)
from huggingface_hub import login
login()
print('HuggingFace 登录成功')

HuggingFace 登录成功


## 0.5 模型预下载

将 Qwen、Llama、roberta-large 预下载到本地 HF cache。
后续训练/推理/评估全部从本地加载，不再依赖网络。

- **Qwen / Llama**：训练和推理使用（通过 HF cache 自动解析）
- **roberta-large**：BERTScore 评估使用（通过 `--bertscore_model` 指向本地路径）

In [ ]:
import os
os.environ['HF_HOME'] = '/root/autodl-tmp/hf_cache'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from huggingface_hub import snapshot_download

print("下载 Qwen2.5-1.5B-Instruct ...")
snapshot_download("Qwen/Qwen2.5-1.5B-Instruct",
                  cache_dir="/root/autodl-tmp/hf_cache",
                  resume_download=True)
print("  ✓ Qwen 完成")

print("下载 Llama-3.2-1B-Instruct ...")
snapshot_download("meta-llama/Llama-3.2-1B-Instruct",
                  cache_dir="/root/autodl-tmp/hf_cache",
                  resume_download=True)
print("  ✓ Llama 完成")

In [14]:
import os, glob

print("=" * 60)
print("路径与模型检查")
print("=" * 60)
# 在路径检查 cell 的开头加一行：
BERTSCORE_MODEL_PATH = '/root/autodl-tmp/roberta-large'
# 1. 检查 HF cache 中的模型
hf_cache = '/root/autodl-tmp/hf_cache'
expected_models = [
    'models--Qwen--Qwen2.5-1.5B-Instruct',
    'models--meta-llama--Llama-3.2-1B-Instruct',
    'models--roberta-large',
]
for m in expected_models:
    model_dir = os.path.join(hf_cache, m)
    snapshots = os.path.join(model_dir, 'snapshots')
    if os.path.isdir(snapshots):
        snap_dirs = os.listdir(snapshots)
        if snap_dirs:
            snap_path = os.path.join(snapshots, snap_dirs[0])
            files = os.listdir(snap_path)
            has_weights = any(f.endswith(('.safetensors', '.bin')) for f in files)
            size_mb = sum(os.path.getsize(os.path.join(snap_path, f))
                         for f in files if os.path.isfile(os.path.join(snap_path, f))) / 1e6
            status = '✓' if has_weights else '⚠ 缺少权重文件'
            print(f'  {status} {m}')
            print(f'      路径: {snap_path}')
            print(f'      大小: {size_mb:.0f} MB, 文件: {len(files)}')
            if m == 'models--roberta-large':
                BERTSCORE_MODEL_PATH = snap_path
        else:
            print(f'  ✗ {m}: snapshots 目录为空')
    else:
        print(f'  ✗ {m}: 未找到')

# 2. 检查项目目录结构
print()
for p in ['configs', 'src/train', 'src/evaluate', 'src/data/pubmed']:
    full = os.path.join('/root/MLP', p)
    status = '✓' if os.path.isdir(full) else '✗'
    print(f'  {status} {p}/')

# 3. 检查关键脚本
for f in ['src/train/train.py', 'src/evaluate/inference.py', 'src/evaluate/eval_billsum.py']:
    full = os.path.join('/root/MLP', f)
    status = '✓' if os.path.isfile(full) else '✗'
    print(f'  {status} {f}')
    
if 'BERTSCORE_MODEL_PATH' not in dir() or not os.path.isdir(BERTSCORE_MODEL_PATH):
    fallback = '/root/autodl-tmp/roberta-large'
    if os.path.isdir(fallback) and any(f.endswith('.safetensors') for f in os.listdir(fallback)):
        BERTSCORE_MODEL_PATH = fallback
        print(f'  ✓ roberta-large (wget 下载): {fallback}')
    else:
        BERTSCORE_MODEL_PATH = 'roberta-large'
        print(f'  ⚠ roberta-large 未找到本地副本，将使用在线下载: {BERTSCORE_MODEL_PATH}')

# 4. 导出 BERTScore 模型路径供后续使用
print(f'\nBERTScore 模型路径: {BERTSCORE_MODEL_PATH}')
print("\n✅ 所有检查通过" if all(
    os.path.isdir(os.path.join(hf_cache, m, 'snapshots'))
    for m in expected_models
) else "\n⚠ 有模型缺失，请检查上方输出")

路径与模型检查
  ✓ models--Qwen--Qwen2.5-1.5B-Instruct
      路径: /root/autodl-tmp/hf_cache/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306
      大小: 3099 MB, 文件: 10
  ✓ models--meta-llama--Llama-3.2-1B-Instruct
      路径: /root/autodl-tmp/hf_cache/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6
      大小: 2481 MB, 文件: 11
  ✗ models--roberta-large: 未找到

  ✓ configs/
  ✓ src/train/
  ✓ src/evaluate/
  ✓ src/data/pubmed/
  ✓ src/train/train.py
  ✓ src/evaluate/inference.py
  ✓ src/evaluate/eval_billsum.py

BERTScore 模型路径: /root/autodl-tmp/roberta-large

⚠ 有模型缺失，请检查上方输出


## 1. 数据下载与格式化

从 HuggingFace 下载 `ccdv/pubmed-summarization`，采样 20k 训练集，格式化为 SFT JSONL。

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/data/pubmed/pubmed_formatting.py --train_samples 20000 --output_dir data/pubmed
print('数据格式化完成')

In [11]:
# 验证数据文件
import os, json
files = [
    'data/pubmed/train_sft.jsonl',
    'data/pubmed/val_sft.jsonl',
    'data/pubmed/test_sft.jsonl',
]
for f in files:
    if os.path.exists(f):
        with open(f) as fh:
            n = sum(1 for _ in fh)
        size = os.path.getsize(f) // (1024 * 1024)
        print(f'✓ {f}: {n:,} records, {size} MB')
    else:
        print(f'✗ {f}: 缺失！')

# 看第一条数据
with open('data/pubmed/train_sft.jsonl') as f:
    sample = json.loads(f.readline())
print(f'\n--- Sample ---')
print(f'Input length: {len(sample["input"])} chars')
print(f'Output length: {len(sample["output"])} chars')
print(f'Output preview: {sample["output"][:200]}...')

✓ data/pubmed/train_sft.jsonl: 20,000 records, 439 MB
✓ data/pubmed/val_sft.jsonl: 6,633 records, 148 MB
✓ data/pubmed/test_sft.jsonl: 6,658 records, 149 MB

--- Sample ---
Input length: 12094 chars
Output length: 1575 chars
Output preview: background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition status among school - aged children in shiraz , iran...


---
## 2. LoRA 实验

### E15: LoRA — PubMed × Qwen2.5-1.5B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
os.makedirs('logs', exist_ok=True)
!{sys.executable} -u src/train/train.py --config configs/lora_pubmed_qwen.yaml 2>&1 | tee logs/lora_pubmed_qwen.log
print('训练完成')

In [ ]:
import sys, os, subprocess
os.chdir('/root/MLP')
os.makedirs('results/pubmed', exist_ok=True)

!{sys.executable} -u src/evaluate/inference.py --config configs/lora_pubmed_qwen.yaml --split test --batch_size 8

In [19]:
import sys, os, subprocess
subprocess.run([sys.executable, '-u', 'src/evaluate/eval_billsum.py',
                '--predictions', 'outputs/lora_pubmed_qwen/predictions_test.jsonl',
                '--output', 'results/pubmed/lora_qwen_test.json',
                '--bertscore_model', BERTSCORE_MODEL_PATH], check=True)
print('评估完成')
!cat results/pubmed/lora_qwen_test.json

Evaluating 6658 samples...


/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 3587.36it/s]
RobertaModel LOAD REPORT from: /root/autodl-tmp/roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
po


BillSum Evaluation Results
  rouge1               0.4103 ± 0.1015
  rouge2               0.1635 ± 0.1009
  rougeL               0.2472 ± 0.0896
  bertscore_f1         0.8650 ± 0.0208

Results saved → results/pubmed/lora_qwen_test.json
评估完成
{
  "rouge1": {
    "mean": 0.41028295038109497,
    "std": 0.10153982667154589
  },
  "rouge2": {
    "mean": 0.1635256523584333,
    "std": 0.10091695900986075
  },
  "rougeL": {
    "mean": 0.24715018051847185,
    "std": 0.08960417178621197
  },
  "bertscore_f1": {
    "mean": 0.8649832494679999,
    "std": 0.020751325010708987
  }
}

### E16: LoRA — PubMed × Llama-3.2-1B

In [20]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/lora_pubmed_llama.yaml 2>&1 | tee logs/lora_pubmed_llama.log
print('训练完成')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/lora_pubmed_llama.yaml
Model   : meta-llama/Llama-3.2-1B-Instruct
Train   : /root/MLP/data/pubmed/train_sft.jsonl  (20,000 records)
Val     : /root/MLP/data/pubmed/val_sft.jsonl
Max input length : 2048
Max output length: 512
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 286.76it/s]
trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750

Preparing datasets...
Tokenizing: 100%|

In [21]:
import sys, os, subprocess
os.chdir('/root/MLP')

!{sys.executable} -u src/evaluate/inference.py --config configs/lora_pubmed_llama.yaml --split test --batch_size 8

subprocess.run([sys.executable, '-u', 'src/evaluate/eval_billsum.py',
                '--predictions', 'outputs/lora_pubmed_llama/predictions_test.jsonl',
                '--output', 'results/pubmed/lora_llama_test.json',
                '--bertscore_model', BERTSCORE_MODEL_PATH], check=True)
print('评估完成')
!cat results/pubmed/lora_llama_test.json

Config     : configs/lora_pubmed_llama.yaml
Task       : summarization
Split      : test  →  /root/MLP/data/pubmed/test_sft.jsonl
Output     : /root/MLP/outputs/lora_pubmed_llama
Batch size : 8
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading base model: meta-llama/Llama-3.2-1B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 296.15it/s]
Loading LoRA adapter: /root/MLP/outputs/lora_pubmed_llama/final_adapter
Loaded 6,658 test records
The following generation flags are not va

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 7736.11it/s]
RobertaModel LOAD REPORT from: /root/autodl-tmp/roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
po


BillSum Evaluation Results
  rouge1               0.4115 ± 0.1089
  rouge2               0.1598 ± 0.1116
  rougeL               0.2417 ± 0.0978
  bertscore_f1         0.8517 ± 0.0248

Results saved → results/pubmed/lora_llama_test.json
评估完成
{
  "rouge1": {
    "mean": 0.4115215111970684,
    "std": 0.10885753049211207
  },
  "rouge2": {
    "mean": 0.15981194644508886,
    "std": 0.11162785721872491
  },
  "rougeL": {
    "mean": 0.24170560853592282,
    "std": 0.09780049256023324
  },
  "bertscore_f1": {
    "mean": 0.8516580036772232,
    "std": 0.02479059004410824
  }
}

---
## 3. Full Fine-Tuning 实验

### E17: Full FT — PubMed × Qwen2.5-1.5B

In [22]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/full_pubmed_qwen.yaml 2>&1 | tee logs/full_pubmed_qwen.log
print('训练完成')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/full_pubmed_qwen.yaml
Model   : Qwen/Qwen2.5-1.5B-Instruct
Train   : /root/MLP/data/pubmed/train_sft.jsonl  (20,000 records)
Val     : /root/MLP/data/pubmed/val_sft.jsonl
Max input length : 2048
Max output length: 512
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 518.14it/s]
Full fine-tuning: 1.54B trainable parameters

Preparing datasets...
Tokenizing: 100%|██████████| 6633/6633 [00:52<00:00, 126.1

In [23]:
import sys, os, subprocess
os.chdir('/root/MLP')

!{sys.executable} -u src/evaluate/inference.py --config configs/full_pubmed_qwen.yaml --split test --batch_size 8

subprocess.run([sys.executable, '-u', 'src/evaluate/eval_billsum.py',
                '--predictions', 'outputs/full_pubmed_qwen/predictions_test.jsonl',
                '--output', 'results/pubmed/full_qwen_test.json',
                '--bertscore_model', BERTSCORE_MODEL_PATH], check=True)
print('评估完成')
!cat results/pubmed/full_qwen_test.json

Config     : configs/full_pubmed_qwen.yaml
Task       : summarization
Split      : test  →  /root/MLP/data/pubmed/test_sft.jsonl
Output     : /root/MLP/outputs/full_pubmed_qwen
Batch size : 8
Loading full fine-tuned model: /root/MLP/outputs/full_pubmed_qwen/final_model
`torch_dtype` is deprecated! Use `dtype` instead!
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|███████████████████████| 338/338 [00:00<00:00, 683.48it/s]
Loaded 6,658 test records
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 7729.62it/s]
RobertaModel LOAD REPORT from: /root/autodl-tmp/roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
po


BillSum Evaluation Results
  rouge1               0.4078 ± 0.1012
  rouge2               0.1612 ± 0.0990
  rougeL               0.2453 ± 0.0884
  bertscore_f1         0.8642 ± 0.0208

Results saved → results/pubmed/full_qwen_test.json
评估完成
{
  "rouge1": {
    "mean": 0.4078375726295532,
    "std": 0.10119166633516614
  },
  "rouge2": {
    "mean": 0.1612472993565856,
    "std": 0.09902562992541146
  },
  "rougeL": {
    "mean": 0.24533076842600185,
    "std": 0.08835365345211446
  },
  "bertscore_f1": {
    "mean": 0.8642499436663235,
    "std": 0.020761135709650658
  }
}

### E18: Full FT — PubMed × Llama-3.2-1B

In [24]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/full_pubmed_llama.yaml 2>&1 | tee logs/full_pubmed_llama.log
print('训练完成')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/full_pubmed_llama.yaml
Model   : meta-llama/Llama-3.2-1B-Instruct
Train   : /root/MLP/data/pubmed/train_sft.jsonl  (20,000 records)
Val     : /root/MLP/data/pubmed/val_sft.jsonl
Max input length : 2048
Max output length: 512
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 300.59it/s]
Full fine-tuning: 1.24B trainable parameters

Preparing datasets...
Tokenizing: 100%|██████████| 6633/6633 [00:44<00:00

In [25]:
import sys, os, subprocess
os.chdir('/root/MLP')

!{sys.executable} -u src/evaluate/inference.py --config configs/full_pubmed_llama.yaml --split test --batch_size 8

subprocess.run([sys.executable, '-u', 'src/evaluate/eval_billsum.py',
                '--predictions', 'outputs/full_pubmed_llama/predictions_test.jsonl',
                '--output', 'results/pubmed/full_llama_test.json',
                '--bertscore_model', BERTSCORE_MODEL_PATH], check=True)
print('评估完成')
!cat results/pubmed/full_llama_test.json

Config     : configs/full_pubmed_llama.yaml
Task       : summarization
Split      : test  →  /root/MLP/data/pubmed/test_sft.jsonl
Output     : /root/MLP/outputs/full_pubmed_llama
Batch size : 8
Loading full fine-tuned model: /root/MLP/outputs/full_pubmed_llama/final_model
`torch_dtype` is deprecated! Use `dtype` instead!
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 298.89it/s]
Loaded 6,658 test records
The following generation flags are not valid and may be ignored: ['temperature', 'top_p

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 6825.60it/s]
RobertaModel LOAD REPORT from: /root/autodl-tmp/roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
po


BillSum Evaluation Results
  rouge1               0.4124 ± 0.1089
  rouge2               0.1609 ± 0.1119
  rougeL               0.2424 ± 0.0985
  bertscore_f1         0.8516 ± 0.0247

Results saved → results/pubmed/full_llama_test.json
评估完成
{
  "rouge1": {
    "mean": 0.41239794192180373,
    "std": 0.10892083643300172
  },
  "rouge2": {
    "mean": 0.16089528168363815,
    "std": 0.11186293447947354
  },
  "rougeL": {
    "mean": 0.2423769864378007,
    "std": 0.09845693188598277
  },
  "bertscore_f1": {
    "mean": 0.8515605708524321,
    "std": 0.02471209754676334
  }
}

---
## 4. Random Label Baseline

将训练集的 output 打乱（input 不变），验证模型是否真正学到了任务知识。

### 4.0 生成 Random Label 数据

In [26]:
import json, random
from pathlib import Path

random.seed(42)

def shuffle_labels(input_path, output_path):
    records = []
    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    outputs = [r['output'] for r in records]
    random.shuffle(outputs)
    for rec, new_out in zip(records, outputs):
        rec['output'] = new_out
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    print(f'  ✓ {output_path}: {len(records):,} records (labels shuffled)')

print('=== Generating Random Label PubMed Datasets ===')
shuffle_labels('data/pubmed/train_sft.jsonl', 'data/pubmed/train_sft_random.jsonl')
shuffle_labels('data/pubmed/val_sft.jsonl',   'data/pubmed/val_sft_random.jsonl')
print('\nDone. Test file is NOT shuffled (evaluate on real data).')

=== Generating Random Label PubMed Datasets ===
  ✓ data/pubmed/train_sft_random.jsonl: 20,000 records (labels shuffled)
  ✓ data/pubmed/val_sft_random.jsonl: 6,633 records (labels shuffled)

Done. Test file is NOT shuffled (evaluate on real data).


In [27]:
# 验证 mismatch rate
import json

def mismatch_rate(orig_path, random_path):
    with open(orig_path) as f1, open(random_path) as f2:
        orig = [json.loads(l)['output'] for l in f1 if l.strip()]
        rand = [json.loads(l)['output'] for l in f2 if l.strip()]
    return sum(a != b for a, b in zip(orig, rand)) / len(orig)

print('Mismatch rates (should be ~1.0):')
print(f'  PubMed train: {mismatch_rate("data/pubmed/train_sft.jsonl", "data/pubmed/train_sft_random.jsonl"):.4f}')
print(f'  PubMed val:   {mismatch_rate("data/pubmed/val_sft.jsonl", "data/pubmed/val_sft_random.jsonl"):.4f}')

Mismatch rates (should be ~1.0):
  PubMed train: 1.0000
  PubMed val:   1.0000


### E19: Random Label LoRA — PubMed × Qwen2.5-1.5B

In [28]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_pubmed_qwen.yaml 2>&1 | tee logs/random_pubmed_qwen.log
print('训练完成')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/random_pubmed_qwen.yaml
Model   : Qwen/Qwen2.5-1.5B-Instruct
Train   : /root/MLP/data/pubmed/train_sft_random.jsonl  (20,000 records)
Val     : /root/MLP/data/pubmed/val_sft_random.jsonl
Max input length : 2048
Max output length: 512
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 549.05it/s]
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

Preparing datasets...
Tokenizi

In [29]:
import sys, os, subprocess
os.chdir('/root/MLP')

!{sys.executable} -u src/evaluate/inference.py --config configs/random_pubmed_qwen.yaml --split test --batch_size 8

subprocess.run([sys.executable, '-u', 'src/evaluate/eval_billsum.py',
                '--predictions', 'outputs/random_pubmed_qwen/predictions_test.jsonl',
                '--output', 'results/pubmed/random_qwen_test.json',
                '--bertscore_model', BERTSCORE_MODEL_PATH], check=True)
print('评估完成')
!cat results/pubmed/random_qwen_test.json

Config     : configs/random_pubmed_qwen.yaml
Task       : summarization
Split      : test  →  /root/MLP/data/pubmed/test_sft.jsonl
Output     : /root/MLP/outputs/random_pubmed_qwen
Batch size : 8
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading base model: Qwen/Qwen2.5-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:00<00:00, 628.12it/s]
Loading LoRA adapter: /root/MLP/outputs/random_pubmed_qwen/final_adapter
Loaded 6,658 test records
The following generation flags are not valid

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 18662.89it/s]
RobertaModel LOAD REPORT from: /root/autodl-tmp/roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
p


BillSum Evaluation Results
  rouge1               0.1900 ± 0.0483
  rouge2               0.0149 ± 0.0147
  rougeL               0.1176 ± 0.0288
  bertscore_f1         0.8078 ± 0.0123

Results saved → results/pubmed/random_qwen_test.json
评估完成
{
  "rouge1": {
    "mean": 0.19002999637516343,
    "std": 0.04832180896523543
  },
  "rouge2": {
    "mean": 0.014887197434100968,
    "std": 0.01474319943953035
  },
  "rougeL": {
    "mean": 0.11761351991330192,
    "std": 0.028774169371014697
  },
  "bertscore_f1": {
    "mean": 0.8077769493530305,
    "std": 0.012330420981134963
  }
}

### E20: Random Label LoRA — PubMed × Llama-3.2-1B

In [30]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_pubmed_llama.yaml 2>&1 | tee logs/random_pubmed_llama.log
print('训练完成')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/random_pubmed_llama.yaml
Model   : meta-llama/Llama-3.2-1B-Instruct
Train   : /root/MLP/data/pubmed/train_sft_random.jsonl  (20,000 records)
Val     : /root/MLP/data/pubmed/val_sft_random.jsonl
Max input length : 2048
Max output length: 512
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 319.35it/s]
trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750

Preparing datasets...
T

In [31]:
import sys, os, subprocess
os.chdir('/root/MLP')

!{sys.executable} -u src/evaluate/inference.py --config configs/random_pubmed_llama.yaml --split test --batch_size 8

subprocess.run([sys.executable, '-u', 'src/evaluate/eval_billsum.py',
                '--predictions', 'outputs/random_pubmed_llama/predictions_test.jsonl',
                '--output', 'results/pubmed/random_llama_test.json',
                '--bertscore_model', BERTSCORE_MODEL_PATH], check=True)
print('评估完成')
!cat results/pubmed/random_llama_test.json

Config     : configs/random_pubmed_llama.yaml
Task       : summarization
Split      : test  →  /root/MLP/data/pubmed/test_sft.jsonl
Output     : /root/MLP/outputs/random_pubmed_llama
Batch size : 8
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading base model: meta-llama/Llama-3.2-1B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 302.72it/s]
Loading LoRA adapter: /root/MLP/outputs/random_pubmed_llama/final_adapter
Loaded 6,658 test records
The following generation flags are 

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 7834.55it/s]
RobertaModel LOAD REPORT from: /root/autodl-tmp/roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
po


BillSum Evaluation Results
  rouge1               0.2209 ± 0.0626
  rouge2               0.0294 ± 0.0209
  rougeL               0.1437 ± 0.0354
  bertscore_f1         0.8043 ± 0.0129

Results saved → results/pubmed/random_llama_test.json
评估完成
{
  "rouge1": {
    "mean": 0.22085640773403728,
    "std": 0.06258274211853505
  },
  "rouge2": {
    "mean": 0.029449853973052505,
    "std": 0.020930737509811153
  },
  "rougeL": {
    "mean": 0.1436937958965616,
    "std": 0.035412527778895274
  },
  "bertscore_f1": {
    "mean": 0.8042567150035611,
    "std": 0.012931274364355079
  }
}

---
## 5. Baseline (Zero-Shot) 实验

不做任何训练，直接用预训练模型在测试集上推理，建立 zero-shot 性能基线。
将预训练模型快照 symlink 到 `outputs/baseline_pubmed_*/final_model/`，让 `inference.py` 直接加载。

In [32]:
import os

hf_cache = '/root/autodl-tmp/hf_cache'

# 找到 Qwen 和 Llama 的 snapshot 路径
def find_snapshot(model_cache_name):
    snap_dir = os.path.join(hf_cache, model_cache_name, 'snapshots')
    snaps = os.listdir(snap_dir)
    return os.path.join(snap_dir, snaps[0])

snap_qwen  = find_snapshot('models--Qwen--Qwen2.5-1.5B-Instruct')
snap_llama = find_snapshot('models--meta-llama--Llama-3.2-1B-Instruct')

# Qwen baseline — symlink 预训练模型到 final_model
os.makedirs('outputs/baseline_pubmed_qwen/final_model', exist_ok=True)
for f in os.listdir(snap_qwen):
    src = os.path.join(snap_qwen, f)
    dst = os.path.join('outputs/baseline_pubmed_qwen/final_model', f)
    if not os.path.exists(dst):
        os.symlink(src, dst)

# Llama baseline — symlink 预训练模型到 final_model
os.makedirs('outputs/baseline_pubmed_llama/final_model', exist_ok=True)
for f in os.listdir(snap_llama):
    src = os.path.join(snap_llama, f)
    dst = os.path.join('outputs/baseline_pubmed_llama/final_model', f)
    if not os.path.exists(dst):
        os.symlink(src, dst)

print(f'Qwen snapshot:  {snap_qwen}')
print(f'Llama snapshot: {snap_llama}')
print('✓ Baseline 模型 symlink 准备完成')

Qwen snapshot:  /root/autodl-tmp/hf_cache/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306
Llama snapshot: /root/autodl-tmp/hf_cache/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6
✓ Baseline 模型 symlink 准备完成


### B3: Baseline (Zero-Shot) — PubMed × Qwen2.5-1.5B

In [33]:
import sys, os, subprocess
os.chdir('/root/MLP')

# 推理（无训练，直接用预训练模型）
!{sys.executable} -u src/evaluate/inference.py --config configs/baseline_pubmed_qwen.yaml --split test --batch_size 8

# 评估
subprocess.run([sys.executable, '-u', 'src/evaluate/eval_billsum.py',
                '--predictions', 'outputs/baseline_pubmed_qwen/predictions_test.jsonl',
                '--output', 'results/pubmed/baseline_qwen_test.json',
                '--bertscore_model', BERTSCORE_MODEL_PATH], check=True)
print('Qwen baseline 评估完成')
!cat results/pubmed/baseline_qwen_test.json

Config     : configs/baseline_pubmed_qwen.yaml
Task       : summarization
Split      : test  →  /root/MLP/data/pubmed/test_sft.jsonl
Output     : /root/MLP/outputs/baseline_pubmed_qwen
Batch size : 8
Loading full fine-tuned model: /root/MLP/outputs/baseline_pubmed_qwen/final_model
`torch_dtype` is deprecated! Use `dtype` instead!
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|███████████████████████| 338/338 [00:00<00:00, 359.04it/s]
Loaded 6,658 test records
The following generation flags are not valid and may be ignored: ['temperature

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 7827.82it/s]
RobertaModel LOAD REPORT from: /root/autodl-tmp/roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
po


BillSum Evaluation Results
  rouge1               0.3705 ± 0.0720
  rouge2               0.1040 ± 0.0505
  rougeL               0.1891 ± 0.0428
  bertscore_f1         0.8292 ± 0.0154

Results saved → results/pubmed/baseline_qwen_test.json
Qwen baseline 评估完成
{
  "rouge1": {
    "mean": 0.3705423999950608,
    "std": 0.0719515683621715
  },
  "rouge2": {
    "mean": 0.10395021130275915,
    "std": 0.05049916390543058
  },
  "rougeL": {
    "mean": 0.18914377978216573,
    "std": 0.04279820234868105
  },
  "bertscore_f1": {
    "mean": 0.8291743861533171,
    "std": 0.015439613958364065
  }
}

### B4: Baseline (Zero-Shot) — PubMed × Llama-3.2-1B

In [34]:
import sys, os, subprocess
os.chdir('/root/MLP')

# 推理（无训练，直接用预训练模型）
!{sys.executable} -u src/evaluate/inference.py --config configs/baseline_pubmed_llama.yaml --split test --batch_size 8

# 评估
subprocess.run([sys.executable, '-u', 'src/evaluate/eval_billsum.py',
                '--predictions', 'outputs/baseline_pubmed_llama/predictions_test.jsonl',
                '--output', 'results/pubmed/baseline_llama_test.json',
                '--bertscore_model', BERTSCORE_MODEL_PATH], check=True)
print('Llama baseline 评估完成')
!cat results/pubmed/baseline_llama_test.json

Config     : configs/baseline_pubmed_llama.yaml
Task       : summarization
Split      : test  →  /root/MLP/data/pubmed/test_sft.jsonl
Output     : /root/MLP/outputs/baseline_pubmed_llama
Batch size : 8
Loading full fine-tuned model: /root/MLP/outputs/baseline_pubmed_llama/final_model
`torch_dtype` is deprecated! Use `dtype` instead!
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 210.72it/s]
Loaded 6,658 test records
The following generation flags are not valid and may be ignored: ['temperat

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 7625.90it/s]
RobertaModel LOAD REPORT from: /root/autodl-tmp/roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
po


BillSum Evaluation Results
  rouge1               0.4123 ± 0.0824
  rouge2               0.1400 ± 0.0696
  rougeL               0.2282 ± 0.0576
  bertscore_f1         0.8398 ± 0.0179

Results saved → results/pubmed/baseline_llama_test.json
Llama baseline 评估完成
{
  "rouge1": {
    "mean": 0.4122545832521303,
    "std": 0.08240551215559733
  },
  "rouge2": {
    "mean": 0.13995151526004082,
    "std": 0.06956035081261776
  },
  "rougeL": {
    "mean": 0.22823403815125032,
    "std": 0.05761915185348439
  },
  "bertscore_f1": {
    "mean": 0.8397657626937123,
    "std": 0.017868199471883617
  }
}

---
## 6. 汇总所有 PubMed 结果

In [35]:
import json, os

results = [
    ('LoRA  × Qwen',        'results/pubmed/lora_qwen_test.json'),
    ('LoRA  × Llama',       'results/pubmed/lora_llama_test.json'),
    ('Full FT × Qwen',     'results/pubmed/full_qwen_test.json'),
    ('Full FT × Llama',    'results/pubmed/full_llama_test.json'),
    ('Random × Qwen',      'results/pubmed/random_qwen_test.json'),
    ('Random × Llama',     'results/pubmed/random_llama_test.json'),
    ('Baseline × Qwen',    'results/pubmed/baseline_qwen_test.json'),
    ('Baseline × Llama',   'results/pubmed/baseline_llama_test.json'),
]

print(f'{"实验":<25} {"ROUGE-1":>10} {"ROUGE-2":>10} {"ROUGE-L":>10} {"BERTScore":>10}')
print('-' * 70)
for name, path in results:
    if not os.path.exists(path):
        print(f'{name:<25} {"未完成":>10}')
        continue
    with open(path) as f:
        d = json.load(f)
    r1 = d['rouge1']['mean'] if isinstance(d['rouge1'], dict) else d['rouge1']
    r2 = d['rouge2']['mean'] if isinstance(d['rouge2'], dict) else d['rouge2']
    rl = d['rougeL']['mean'] if isinstance(d['rougeL'], dict) else d['rougeL']
    bs = d['bertscore_f1']['mean'] if isinstance(d['bertscore_f1'], dict) else d['bertscore_f1']
    print(f'{name:<25} {r1:>10.4f} {r2:>10.4f} {rl:>10.4f} {bs:>10.4f}')

实验                           ROUGE-1    ROUGE-2    ROUGE-L  BERTScore
----------------------------------------------------------------------
LoRA  × Qwen                  0.4103     0.1635     0.2472     0.8650
LoRA  × Llama                 0.4115     0.1598     0.2417     0.8517
Full FT × Qwen                0.4078     0.1612     0.2453     0.8642
Full FT × Llama               0.4124     0.1609     0.2424     0.8516
Random × Qwen                 0.1900     0.0149     0.1176     0.8078
Random × Llama                0.2209     0.0294     0.1437     0.8043
Baseline × Qwen               0.3705     0.1040     0.1891     0.8292
Baseline × Llama              0.4123     0.1400     0.2282     0.8398
